In [ ]:
# Load Doc

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

loader=PyPDFLoader("10 RAG Architectures.pdf")
documents=loader.load()

splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=splitter.split_documents(documents)

#load Embedding model
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed all Chunks and store in FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

# Save to disk so you dont Re-embed every time
vectorstore.save_local("faiss_index")

# Load later
vectorstore= FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# Create retriever from vectorstore
retriver=vectorstore.as_retriever(
    search_kwargs={"k":3} # Return top three most similar chunks
)

# Retrieve for a query
query= "name some RAG architecture"
relevant_chunks= retriver.invoke(query)

# See what was retrieved
for chunk in relevant_chunks:
    print(chunk.page_content)
    print("---")

# Final step
Combine retrieve chunks + User question into a prompt, sent to LLM, get grounded answer.

In [6]:
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

# Load LLM
llm= ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# Prompt template
prompt_template= """ 
You are an expert on rag architectures.
Use only the context below to answer the question.
If the answer is not in the context ,Say "I don't know".

Context: {context}
Question: {question}
Answer:
"""

# Build RAG chain
rag_chain= RetrievalQA.from_chain_type(
    llm=llm,
    retriver= retriver,
    chain_type_kwargs={"prompt": prompt}
)

# Ask question about your PDF
questions = [
    " What is standard RAG?",
    " Name all 10 RAG architectures",
    " Work is the difference between Standard RAG and agentic RAG?"
] 

for q in questions:
    print(f"\nQ: {q}")
    result = rag_chain.invoke({"query": q})
    print(f"A: {result['result']}")
    print("-"*50)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

# Key things in the prompt
→"Use ONLY the context below" — grounds the LLM, prevents hallucination.

→"If not in context say I don't know" — prevents making things up.

→{context} gets filled with retrieved chunks automatically.

→{question} gets filled with user's query.